In [ ]:
# =============================================================================
# TASK 1A - RGB Land-Use Classification | EfficientNet-B0
# Geo Snap Competition (SOI x Cosmosoc)
# Dataset : EuroSAT RGB 64x64 JPG | 10 land-use classes
# Output  : rgb_predictions.csv (img_id, predicted_label)
# Hardware: RTX 5060 24GB
# =============================================================================

import os, json, time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
CFG = {
    "train_csv"    : r"EuroSAT_Dataset/train.csv",
    "val_csv"      : r"EuroSAT_Dataset/validation.csv",
    "label_map"    : r"EuroSAT_Dataset/label_map.json",
    "train_dir"    : r"EuroSAT_Dataset/EuroSAT/train", 
    "val_dir"      : r"EuroSAT_Dataset/EuroSAT/val",  
    "test_dir"     : r"EuroSAT_Dataset/EuroSAT_test_flat",
    "num_classes"  : 10,
    "img_size"     : 64,
    "batch_size"   : 128,   # RTX 5060 24GB can handle large batches easily
    "epochs"       : 100,
    "lr"           : 1e-3,
    "weight_decay" : 1e-4,
    "patience"     : 7,
    "num_workers"  : 0,     # Safest for Windows Jupyter
    "save_path"    : "best_rgb_model.pth",
    "pred_csv"     : "rgb_predictions.csv",
    "plots_dir"    : "plots",
}

# Select GPU if available, otherwise use CPU.
# Training on GPU significantly reduces training time.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.makedirs(CFG["plots_dir"], exist_ok=True)

# enable tf32 on Ampere/Ada GPUs - faster matmuls with no real accuracy loss
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True  # auto-tune conv algorithms for your GPU

# List of land cover classes in the same order as the dataset labels.
CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial", 
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"
]

# ── Transforms ────────────────────────────────────────────────────────────────
# ImageNet mean/std used because EfficientNet-B0 was pretrained on ImageNet
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.RandomHorizontalFlip(),   # randomly flip images horizontally
    transforms.RandomVerticalFlip(),    # satellite has no fixed orientation
    transforms.RandomRotation(15),    # improves rotational invariance
    transforms.ColorJitter(0.2, 0.2, 0.1),  # simulate lighting / seasonal variation
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset ───────────────────────────────────────────────────────────────────

# Loads RGB satellite images together with their labels for training
# and validation. The same class is also reused for test inference
# where ground truth labels are unavailable.

class EuroSATRGB(Dataset):
    """
    Train/Val mode : reads filenames + labels from CSV
    Test mode      : reads all images from flat folder (no labels)
    """
    def __init__(self, csv_file=None, img_dir=None, tf=None,
                 test_dir=None, label_map=None):
        self.tf        = tf
        self.label_map = label_map
        self.test_mode = test_dir is not None

        if self.test_mode:
            self.img_dir = test_dir
            self.files   = sorted(f for f in os.listdir(test_dir)
                                  if f.lower().endswith(('.jpg', '.png')))
        else:
            self.img_dir = img_dir
            df           = pd.read_csv(csv_file)
            self.files   = df['filename'].tolist()
            
            self.labels  = df['label'].tolist()

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):

        # Convert Windows-style paths to a consistent format
        # to avoid file loading issues.

        filename = self.files[idx].replace("\\", "/")
        img = Image.open(os.path.join(self.img_dir, filename)).convert("RGB")
        if self.tf: img = self.tf(img)
        if self.test_mode:
            return img, self.files[idx]
        return img, self.labels[idx]

# ── Model ─────────────────────────────────────────────────────────────────────
def build_model():
    # pretrained EfficientNet-B0 on ImageNet
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    for p in m.parameters(): p.requires_grad = False        # Freeze all pretrained layers to preserve learned visual features.
    for i in [6, 7, 8]:                                     # Fine-tune only the last few feature extraction blocks.
        for p in m.features[i].parameters(): p.requires_grad = True

    # replace head: 1280 → 10 classes
    m.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(m.classifier[1].in_features, CFG["num_classes"])
    )
    return m

# ── Train / Validate ──────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    """Single train or val epoch. optimizer=None → validation mode."""
    training = optimizer is not None
    model.train() if training else model.eval()

    loss_sum, correct, total = 0, 0, 0
    preds_all, true_all = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            if training and scaler:
                # AMP: mixed precision - runs in fp16 where possible, much faster on RTX
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    out  = model(imgs)
                    loss = criterion(out, labels)
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out  = model(imgs)
                loss = criterion(out, labels)
                if training:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

            loss_sum += loss.item() * imgs.size(0)
            preds     = out.argmax(1)
            correct  += (preds == labels).sum().item()
            total    += imgs.size(0)
            preds_all.extend(preds.cpu().tolist())
            true_all.extend(labels.cpu().tolist())

    return loss_sum / total, correct / total, preds_all, true_all

# ── Plots ─────────────────────────────────────────────────────────────────────
def save_curves(tl, vl, ta, va):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    ep = range(1, len(tl)+1)
    a1.plot(ep, tl, label='Train'); a1.plot(ep, vl, label='Val')
    a1.set(title='Loss', xlabel='Epoch'); a1.legend(); a1.grid(alpha=0.3)
    a2.plot(ep, [x*100 for x in ta], label='Train')
    a2.plot(ep, [x*100 for x in va], label='Val')
    a2.set(title='Accuracy (%)', xlabel='Epoch'); a2.legend(); a2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CFG['plots_dir']}/curves_rgb.png", dpi=150); plt.close()

def save_cm(true, pred):
    cm = confusion_matrix(true, pred).astype(float)
    cm /= cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix (Normalized) - RGB Val Set')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig(f"{CFG['plots_dir']}/cm_rgb.png", dpi=150); plt.close()

# =============================================================================
# Main training pipeline
#
# Loads the dataset, initializes the model, trains the network,
# evaluates the best checkpoint, generates plots, and performs
# inference on the test dataset.
# =============================================================================
# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    label_map = json.load(open(CFG["label_map"]))

    train_ds = EuroSATRGB(CFG["train_csv"], CFG["train_dir"], train_tf, label_map=label_map)
    val_ds   = EuroSATRGB(CFG["val_csv"],   CFG["val_dir"],   val_tf,   label_map=label_map)

    train_dl = DataLoader(train_ds, CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True)
    val_dl   = DataLoader(val_ds,   CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
    # pin_memory=True - locks data in RAM for faster GPU transfer

    model     = build_model().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    # Reduce the learning rate automatically when the validation
    # accuracy stops improving. This helps the model converge better.
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    # AMP GradScaler - handles fp16 gradient scaling automatically
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    best_acc, wait = 0.0, 0
    tl, vl, ta, va = [], [], [], []

    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Batch: {CFG['batch_size']}\n")

    for ep in range(1, CFG["epochs"]+1):
        t0 = time.time()
        tr_loss, tr_acc, _, _ = run_epoch(model, train_dl, criterion, optimizer, scaler)
        v_loss,  v_acc, vp, vt = run_epoch(model, val_dl, criterion)
        scheduler.step(v_acc)

        tl.append(tr_loss); vl.append(v_loss)
        ta.append(tr_acc);  va.append(v_acc)

        print(f"Ep {ep:02d} [{time.time()-t0:.0f}s]  "
              f"Train {tr_loss:.3f}/{tr_acc*100:.1f}%  "
              f"Val {v_loss:.3f}/{v_acc*100:.1f}%")

        if v_acc > best_acc:
            best_acc = v_acc
            torch.save(model.state_dict(), CFG["save_path"])
            print(f"  -> saved best ({best_acc*100:.2f}%)")
            wait = 0
        else:
            wait += 1
            if wait >= CFG["patience"]: print("Early stop."); break

    # ── Final eval ────────────────────────────────────────────────────────────
    # Evaluate the best saved model on the validation set
    # and generate performance reports.
    model.load_state_dict(torch.load(CFG["save_path"], map_location=DEVICE, weights_only=True))
    _, acc, preds, true = run_epoch(model, val_dl, criterion)
    print(f"\nFinal Val Acc: {acc*100:.2f}%")
    print(classification_report(true, preds, target_names=CLASS_NAMES, digits=3))
    save_curves(tl, vl, ta, va)
    save_cm(true, preds)

    # ── Test inference ────────────────────────────────────────────────────────
    # Run inference on unseen test images and save
    # the predicted class labels into a CSV file.
    test_ds = EuroSATRGB(test_dir=CFG["test_dir"], tf=val_tf)
    test_dl = DataLoader(test_ds, CFG["batch_size"], shuffle=False,
                         num_workers=CFG["num_workers"], pin_memory=True)
    model.eval()
    ids, out_preds = [], []
    with torch.no_grad():
        for imgs, names in test_dl:
            ids.extend(names)
            out_preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().tolist())

    pd.DataFrame({
        "img_id"         : ids,
        "predicted_label": [CLASS_NAMES[p] for p in out_preds]
    }).to_csv(CFG["pred_csv"], index=False)

    print(f"\nPredictions -> {CFG['pred_csv']}")
    print(f"Plots       -> {CFG['plots_dir']}/")

if __name__ == "__main__":
    main() 

[INFO] Device: cuda
[INFO] GPU: NVIDIA GeForce RTX 5060 Laptop GPU
[INFO] VRAM: 8.5 GB
Train: 18900 | Val: 4050 | Batch: 128

Ep 01 [196s]  Train 0.582/80.7%  Val 0.308/90.0%
  -> saved best (90.00%)
Ep 02 [77s]  Train 0.326/88.8%  Val 0.222/93.1%
  -> saved best (93.09%)
Ep 03 [74s]  Train 0.276/90.6%  Val 0.188/93.9%
  -> saved best (93.85%)
Ep 04 [73s]  Train 0.249/91.5%  Val 0.198/93.2%
Ep 05 [73s]  Train 0.236/92.1%  Val 0.177/94.1%
  -> saved best (94.10%)
Ep 06 [74s]  Train 0.221/92.5%  Val 0.165/94.9%
  -> saved best (94.94%)
Ep 07 [74s]  Train 0.215/92.8%  Val 0.179/94.3%
Ep 08 [74s]  Train 0.199/93.2%  Val 0.172/94.2%
Ep 09 [77s]  Train 0.187/93.5%  Val 0.179/94.6%
Ep 10 [75s]  Train 0.187/93.5%  Val 0.162/94.7%
Ep 11 [74s]  Train 0.153/94.9%  Val 0.150/95.4%
  -> saved best (95.41%)
Ep 12 [73s]  Train 0.148/94.6%  Val 0.152/95.1%
Ep 13 [74s]  Train 0.140/95.0%  Val 0.156/95.0%
Ep 14 [74s]  Train 0.146/94.8%  Val 0.138/95.4%
Ep 15 [76s]  Train 0.136/95.4%  Val 0.145/95.3%
Ep 